# Now with attention

This notebook starts from almost the same recurrent model as `rnn-limits.ipynb`, but adds one small causal self-attention layer on top of the recurrent hidden states. The point is to show that attention gives each timestep a learned distribution over earlier context, instead of forcing all long-range influence through a steeply shrinking recurrent chain.

In [7]:
import os
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

seed = 7
random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cpu")

## Load the hard-wrapped Oz corpus

We will keep the data pipeline the same as before: one hard-wrapped nonempty line per training example, character-level vocabulary, and newline as the natural stopping token for generation.

In [8]:
files = sorted(Path("data/oz").glob("*.txt"))
raw_lines = []
for path in files:
    raw_lines.extend(path.read_text(encoding="utf-8").splitlines())

lines = [line.strip() for line in raw_lines if line.strip()]
max_len = 80
lines = [line[: max_len - 1] + "\n" for line in lines]

chars = sorted(set("".join(lines)))
pad_token = "<PAD>"
vocab = [pad_token] + chars
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
pad_id = stoi[pad_token]
vocab_size = len(vocab)
seq_len = max_len - 1

def encode(text):
    return [stoi[ch] for ch in text]

def decode(ids):
    return "".join(itos[i] for i in ids if i != pad_id)

encoded = torch.full((len(lines), max_len), pad_id, dtype=torch.long)
for i, line in enumerate(lines):
    ids = torch.tensor(encode(line), dtype=torch.long)
    encoded[i, : len(ids)] = ids

x_data = encoded[:, :-1]
y_data = encoded[:, 1:]
train_ds = TensorDataset(x_data, y_data)

print(f"books: {len(files)}")
print(f"nonempty lines: {len(lines):,}")
print(f"vocab size (plus PAD): {vocab_size}")
print(f"training sequence length: {seq_len}")
print()
for sample in lines[:5]:
    print(repr(sample))

books: 14
nonempty lines: 56,131
vocab size (plus PAD): 77
training sequence length: 79

"The train from 'Frisco was very late.  It should have arrived at\n"
"Hugson's Siding at midnight, but it was already five o'clock and the\n"
'gray dawn was breaking in the east when the little train slowly rumbled\n'
'up to the open shed that served for the station-house.  As it came to a\n'
'stop the conductor called out in a loud voice:\n'


## Add one causal self-attention layer on top of the recurrent states

The recurrent part is still simple:

$$
 h_t = \sigma(W_{xh} x_t + W_{hh} h_{t-1} + b_h).
$$

After computing all hidden states, we let each position form a query, key, and value from its hidden state. A causal mask then turns the dot-product scores into a probability distribution over the current and earlier positions only. That distribution is the new context path.

In [9]:
class AttentiveCharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        scale = 0.08
        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * scale)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * scale)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        self.W_q = nn.Parameter(torch.randn(hidden_size, hidden_size) * scale)
        self.W_k = nn.Parameter(torch.randn(hidden_size, hidden_size) * scale)
        self.W_v = nn.Parameter(torch.randn(hidden_size, hidden_size) * scale)

        self.W_hy = nn.Parameter(torch.randn(hidden_size, vocab_size) * scale)
        self.b_y = nn.Parameter(torch.zeros(vocab_size))

    def recurrent_states_from_onehot(self, x_onehot, h0=None):
        batch_size, steps, _ = x_onehot.shape
        if h0 is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x_onehot.device)
        else:
            h = h0
        states = []
        for t in range(steps):
            x_t = x_onehot[:, t, :]
            h = torch.sigmoid(x_t @ self.W_xh + h @ self.W_hh + self.b_h)
            states.append(h)
        return torch.stack(states, dim=1)

    def forward_from_onehot(self, x_onehot, h0=None, return_details=False):
        states = self.recurrent_states_from_onehot(x_onehot, h0=h0)
        q = states @ self.W_q
        k = states @ self.W_k
        v = states @ self.W_v

        scores = q @ k.transpose(1, 2) / math.sqrt(self.hidden_size)
        steps = scores.size(-1)
        causal_mask = torch.triu(torch.ones(steps, steps, device=scores.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal_mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        context = attn @ v

        combined = states + context
        logits = combined @ self.W_hy + self.b_y

        if return_details:
            return logits, states, attn, context
        return logits

    def forward(self, x_ids, h0=None, return_details=False):
        x_onehot = F.one_hot(x_ids, num_classes=vocab_size).float()
        return self.forward_from_onehot(x_onehot, h0=h0, return_details=return_details)

model = AttentiveCharRNN(vocab_size=vocab_size, hidden_size=64).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

AttentiveCharRNN


## Train the model

We still do teacher-forced next-character prediction. The model is only slightly richer than the plain RNN, so the main lesson should still be easy to see: a later prediction can look back through an explicit attention distribution, rather than depending only on the final compressed hidden state.

In [ ]:
batch_size = 256
epochs = 4
loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
loss_history = []

for epoch in range(epochs):
    total_loss = 0.0
    total_tokens = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        token_count = (yb != pad_id).sum().item()
        total_loss += loss.item() * token_count
        total_tokens += token_count
        loss_history.append(loss.item())

    mean_nll = total_loss / total_tokens
    print(f"epoch {epoch + 1}: mean token NLL = {mean_nll:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(loss_history, lw=1)
plt.title("Training loss by mini-batch")
plt.xlabel("mini-batch")
plt.ylabel("cross-entropy")
plt.show()

epoch 1: mean token NLL = 2.8811


epoch 2: mean token NLL = 2.3840


## Sample a few lines

The samples only need to show that the model learned local Oz-like structure. The real difference from the plain RNN will show up in the attention weights, not in dramatic text quality.

In [ ]:
@torch.no_grad()
def generate_line(model, prompt="The ", max_new_chars=60, temperature=0.8):
    model.eval()
    text = prompt
    if prompt:
        ids = torch.tensor([stoi[ch] for ch in prompt], dtype=torch.long, device=device).unsqueeze(0)
    else:
        ids = torch.tensor([[stoi["T"]]], dtype=torch.long, device=device)
        text = "T"

    for _ in range(max_new_chars):
        logits, _, _, _ = model(ids, return_details=True)
        next_logits = logits[0, -1] / temperature
        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        next_char = itos[next_id]
        if next_char == "\n":
            break
        text += next_char
        ids = torch.cat([ids, torch.tensor([[next_id]], dtype=torch.long, device=device)], dim=1)
    return text

for prompt in ["The ", '"I ', "Dorothy ", "Oz "]:
    print(f"{prompt:10s} → {generate_line(model, prompt=prompt)}")

## Visualize self-attention on the first few characters

In the plain RNN notebook, the influence of early characters had to travel through repeated recurrent multiplication, and that influence died away quickly. Here, each row of the attention matrix is an explicit probability distribution over earlier positions. If the model wants to look back, it can learn to do that directly.

In [ ]:
prompt = "The train from 'Frisco"
prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
_, states, attn, context = model(prompt_ids, return_details=True)
attn_map = attn[0].detach().cpu()
labels = list(prompt)

plt.figure(figsize=(8, 8))
plt.imshow(attn_map, cmap="hot", interpolation="nearest")
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.xlabel("key / value positions")
plt.ylabel("query positions")
plt.title("Causal self-attention over `The train from 'Frisco`")
plt.colorbar(label="attention weight")
plt.tight_layout()
plt.show()

row_sums = attn_map.sum(dim=-1)
print("row sums:", [round(x.item(), 4) for x in row_sums[:8]], "...")
print("max future attention:", float(torch.triu(attn_map, diagonal=1).max()))

Each row in the heatmap is a learned context distribution. Unlike the plain RNN's recurrent gradient path, it does not have to decay monotonically with distance. The model can place mass wherever earlier context seems useful, which is the core reason attention changes the long-range information story.